Configure the local data snapshots as described in the README before running this notebook.

# Risk regime detection and forecasting in the Brazilian market

Do regime probabilities improve risk forecasts and net strategy performance for BOVA11? Run the notebook with the local research snapshot configured in the README.

In [ ]:
import sys, subprocess
if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable,"-m","pip","-q","install","exchange-calendars","hmmlearn","arch"])

In [ ]:
from retratos_locais import ler_retrato_local

BASE = ler_retrato_local('Ibovespa_Regimes_de_Risco.ipynb', 'BASE')


In [ ]:
import io,base64,zlib,shutil,json,hashlib,inspect
from importlib.metadata import version
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from scipy.special import logsumexp
from scipy.stats import norm
from hmmlearn.hmm import GaussianHMM
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression
from arch.bootstrap import CircularBlockBootstrap

ATIVO_REGIME = "BOVA11"
CORTE = pd.Timestamp("2025-01-01")
SEED = 42
VOL_ALVO = .12
QUARENTENA_BOVA11 = {'2026-10-26': {'motivo': 'OHLC divergente entre provedores. Abertura oficial ainda não confirmada.', 'abertura_yahoo': 140.0, 'abertura_alternativa': 169.0, 'maxima_alternativa': 176.6, 'minima_alternativa': 165.06, 'fechamento_alternativo': 172.42, 'fonte_alternativa': 'https://www.cotab3.com.br/cotacoes?meses=12&ticker=BOVA11', 'referencia_oficial': 'https://www.b3.com.br/pt_br/market-data-e-indices/servicos-de-dados/market-data/historico/mercado-a-vista/cotacoes-historicas/'}}
CUSTO_LADO = .0015
MULTIPLICADOR_ALTO = .60
PASTA = Path("resultados_regimes")/"P1"/ATIVO_REGIME
PASTA.mkdir(parents=True,exist_ok=True)

def ler(nome):
    conteudo = zlib.decompress(base64.b64decode(BASE[nome]))
    df = pd.read_csv(io.BytesIO(conteudo),index_col=0,parse_dates=True)
    df.index = df.index.astype("datetime64[ns]")
    assert df.index.is_monotonic_increasing and df.index.is_unique
    return df

bova,ibov = ler("BOVA11"),ler("IBOV")
try:
    from calendario_mercado import calendario_e_sessoes
except ModuleNotFoundError as erro:
    if erro.name != "calendario_mercado":
        raise
    import exchange_calendars as xcals

    # Fallback gerado a partir de calendario_mercado.py.
    def calendario_e_sessoes(nome, inicio, fim):
        """Retorna o calendário e os rótulos de pregão entre duas datas inclusivas."""
        inicio, fim = pd.Timestamp(inicio), pd.Timestamp(fim)
        if any(pd.isna(dia) or dia.tz is not None or dia != dia.normalize()
               for dia in (inicio, fim)) or inicio > fim:
            raise ValueError("Informe datas locais, sem horário, em ordem crescente.")
        margem = pd.Timedelta(days=15)
        calendario = xcals.get_calendar(nome, start=inicio - margem, end=fim + margem)
        sessoes = calendario.sessions_in_range(inicio, fim)
        if sessoes.tz is not None:
            sessoes = sessoes.tz_localize(None)
        return calendario, sessoes

cal, sessoes = calendario_e_sessoes("BVMF", ibov.index.min(), ibov.index.max())
assert sessoes.equals(ibov.index) and bova.index.isin(sessoes).all()
assert ATIVO_REGIME in {"BOVA11","IBOV"}
preco = (bova["Adj Close"] if ATIVO_REGIME=="BOVA11" else ibov.Close).reindex(sessoes)
assert np.isfinite(preco.dropna()).all() and preco.dropna().gt(0).all()
r = np.log(preco).diff()
pre = r.loc[r.index<CORTE]
treino = max([s.dropna() for _,s in pre.groupby(pre.isna().cumsum())],key=len)*100
assert len(treino)>=252 and treino.index.max()<CORTE
print("Treino contíguo:",len(treino),"| fim:",treino.index.max().date(),
      "| sessões sem fechamento:",int(preco.isna().sum()))

## 1. Regime detection and next-session forecasts

The Gaussian HMM estimates state-specific means and variances. Markov switching uses a common mean and state-dependent variance. States are ordered by fitted variance.

In [ ]:
hmm = GaussianHMM(n_components=3,covariance_type="diag",n_iter=400,random_state=SEED)
hmm.fit(treino.to_numpy()[:,None])
ms = MarkovRegression(treino.to_numpy(),k_regimes=3,trend="c",
                      switching_trend=False,switching_variance=True)
# Statsmodels recentes aceitam um gerador próprio. Versões anteriores
# usam a inicialização padrão determinística e registram essa diferença.
if "rng" in inspect.signature(ms.fit).parameters:
    modo_ms = "busca_com_rng_explicito"
    ms_fit = ms.fit(search_reps=8,rng=np.random.default_rng(SEED),
                    em_iter=8,maxiter=400,disp=False)
else:
    modo_ms = "inicio_padrao_sem_busca_aleatoria"
    ms_fit = ms.fit(search_reps=0,em_iter=8,maxiter=400,disp=False)
ms_params = dict(zip(ms.param_names,ms_fit.params))
assert ms_fit.mle_retvals["converged"], "Markov Switching sem convergência."
print("HMM critério de parada:",hmm.monitor_.converged,"| Markov Switching convergiu")

try:
    from filtro_regimes import filtrar
except ModuleNotFoundError as erro:
    if erro.name != "filtro_regimes":
        raise
    # Fallback gerado a partir de filtro_regimes.py.
    def filtrar(serie, media, var, A, inicial):
        """Ordena estados por variância e devolve probabilidades atuais e de D+1."""
        observacoes = np.asarray(serie, dtype=float)
        media, var = np.asarray(media, dtype=float), np.asarray(var, dtype=float)
        A, inicial = np.asarray(A, dtype=float), np.asarray(inicial, dtype=float)
        k = var.size
        if (observacoes.ndim != 1 or np.isinf(observacoes).any() or not k
            or media.shape != (k,) or var.shape != (k,)
            or A.shape != (k, k) or inicial.shape != (k,)):
            raise ValueError("Dimensões ou observações inválidas para o filtro de regimes.")
        if (not np.isfinite(media).all() or not np.isfinite(var).all() or (var <= 0).any()
            or not np.isfinite(A).all() or (A < 0).any() or not np.allclose(A.sum(axis=1), 1)
            or not np.isfinite(inicial).all() or (inicial < 0).any() or not np.isclose(inicial.sum(), 1)):
            raise ValueError("Distribuições gaussianas ou probabilidades de transição inválidas.")

        ordem = np.argsort(var)
        media, var = media[ordem], var[ordem]
        A = A[np.ix_(ordem, ordem)]
        prior = inicial[ordem].copy()
        hoje = np.empty((len(observacoes), k))
        amanha = np.empty_like(hoje)
        dispersao = np.sqrt(var)
        for i, valor in enumerate(observacoes):
            if pd.notna(valor):
                logpeso = np.log(np.maximum(prior, 1e-300)) + norm.logpdf(valor, media, dispersao)
                prior = np.exp(logpeso - logsumexp(logpeso))
            hoje[i] = prior
            prior = prior @ A
            amanha[i] = prior
        assert np.allclose(amanha.sum(axis=1), 1)
        return hoje, amanha, media, var, A

ms_A = ms.regime_transition_matrix(ms_fit.params)[:,:,0].T
parametros = {
    "HMM":(hmm.means_.ravel(),hmm.covars_.reshape(3,-1)[:,0],hmm.transmat_,hmm.startprob_),
    "Markov Switching":(np.repeat(ms_params["const"],3),
        np.array([ms_params[f"sigma2[{i}]"] for i in range(3)]),ms_A,ms.initial_probabilities(ms_fit.params))
}
nomes = ["Baixa","Média","Alta"]
probabilidades = {}
def hash_objeto(objeto):
    return hashlib.sha256(json.dumps(objeto,sort_keys=True,allow_nan=False).encode()).hexdigest()

manifesto = {
    "revisao":"P1.1",
    "ativo_regime":ATIVO_REGIME,
    "sha256_snapshot":hashlib.sha256(zlib.decompress(base64.b64decode(BASE[ATIVO_REGIME]))).hexdigest(),
    "inicio_ajuste":str(treino.index.min().date()),
    "fim_ajuste":str(treino.index.max().date()),
    "n_ajuste":len(treino),
    "primeiro_alvo_teste":str(CORTE.date()),
    "semente":SEED,
    "versoes":{nome:version(nome) for nome in
               ["numpy","pandas","scipy","statsmodels","hmmlearn","exchange-calendars","arch"]},
    "modelos":{},
}
# A última origem continua exportável mesmo antes de observar o próximo retorno.
alvos_exportacao = pd.Series(sessoes,index=sessoes).shift(-1)
alvos_exportacao.iloc[-1] = cal.next_session(sessoes[-1])
fora_ajuste = alvos_exportacao.ge(CORTE)&(sessoes>=treino.index.max())
metadados = pd.DataFrame({
    "data_alvo":alvos_exportacao,
    "status_temporal":np.where(fora_ajuste,"fora_da_amostra","retrospectiva_de_ajuste"),
    "alvo_observado":r.shift(-1).notna(),
    "retorno_origem_observado":r.notna(),
    "fim_ajuste":treino.index.max(),
},index=sessoes)
variancias = pd.DataFrame({"Histórica 21":r.pow(2).rolling(21).mean(),
    "EWMA":r.pow(2).groupby(r.isna().cumsum()).transform(
        lambda x:x.ewm(alpha=.06,min_periods=21,adjust=False).mean()).where(r.notna())})
for nome,param in parametros.items():
    atual,futuro,medias,vars_,A = filtrar(r*100,*param)
    # Checagem causal: a trajetória até o corte é igual mesmo sem observações posteriores.
    corte = len(r)//2
    _,curta,*_ = filtrar(r.iloc[:corte]*100,*param)
    assert np.allclose(futuro[:corte],curta)
    print(nome)
    display(pd.DataFrame(A,index=nomes,columns=nomes).round(3))
    display(pd.Series(np.sqrt(vars_*252)/100,index=nomes,name="volatilidade anual de cada estado"))
    probabilidades[nome] = pd.DataFrame(futuro,index=sessoes,columns=nomes)
    variancias[nome] = futuro@(vars_+medias**2)/1e4
    detalhe = {
        "medias_percentuais":medias.tolist(),
        "variancias_percentuais":vars_.tolist(),
        "transicao":A.tolist(),
        "parametros_brutos":ms_params if nome=="Markov Switching" else {
            "inicial":hmm.startprob_.tolist(),"medias":hmm.means_.ravel().tolist(),
            "variancias":hmm.covars_.reshape(3,-1)[:,0].tolist(),
            "transicao":hmm.transmat_.tolist(),
        },
        "log_verossimilhanca":float(ms_fit.llf if nome=="Markov Switching"
                                    else hmm.score(treino.to_numpy()[:,None])),
        "convergencia":bool(ms_fit.mle_retvals["converged"] if nome=="Markov Switching"
                            else hmm.monitor_.converged),
        "inicializacao":modo_ms if nome=="Markov Switching" else "hmm_random_state",
    }
    identidade = hash_objeto({"modelo":nome,"detalhe":detalhe,
        "snapshot":manifesto["sha256_snapshot"],"inicio_ajuste":manifesto["inicio_ajuste"],
        "fim_ajuste":manifesto["fim_ajuste"],"versoes":manifesto["versoes"]})
    manifesto["modelos"][nome] = detalhe|{"id_ajuste":identidade}
    filtrada = pd.DataFrame(atual,index=sessoes,columns=nomes).add_prefix("filtrada_D_")
    exportar = metadados.join(filtrada).join(probabilidades[nome].add_prefix("prevista_D1_"))
    exportar["id_ajuste"] = identidade
    exportar.to_csv(PASTA/f"probabilidades_{nome.replace(' ','_')}.csv",index_label="origem")
(PASTA/"manifesto_regimes.json").write_text(
    json.dumps(manifesto,indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")
fig,axs = plt.subplots(2,1,figsize=(12,6),sharex=True)
for ax,(nome,probs) in zip(axs,probabilidades.items()):
    probs.loc[CORTE:].plot(ax=ax,title=f"{nome} · probabilidade de regime em D+1",ylim=(0,1))
    ax.yaxis.set_major_formatter(PercentFormatter(1))
fig.tight_layout()
fig.savefig(PASTA/"probabilidades_D1.png",dpi=140)
plt.show()

## 2. H1 · Forecast accuracy

Because regimes are latent, the predicted second moment is evaluated against an observable daily proxy. Lower QLIKE is better. The paired bootstrap uses blocks of ten exchange sessions.

In [ ]:
data_alvo = pd.Series(sessoes,index=sessoes).shift(-1)
observado = r.shift(-1).pow(2)
comum = variancias.join(observado.rename("observada")).loc[data_alvo.ge(CORTE)].dropna()
assert len(comum)>40
rows = []
for nome in variancias:
    h = comum[nome].clip(lower=1e-10)
    base = comum.EWMA.clip(lower=1e-10)
    loss = np.log(h)+comum.observada/h
    dif = loss-(np.log(base)+comum.observada/base)
    ic = CircularBlockBootstrap(10,dif.to_numpy(),seed=SEED).conf_int(
        np.mean,reps=2000,method="percentile").ravel()
    rows.append({"modelo":nome,"n":len(comum),"QLIKE":loss.mean(),
        "MAE_variancia":(h-comum.observada).abs().mean(),"dif_vs_EWMA":dif.mean(),
        "IC_inf":ic[0],"IC_sup":ic[1]})
ranking = pd.DataFrame(rows).sort_values("QLIKE")
display(ranking)
ranking.to_csv(PASTA/"previsao_risco.csv",index=False)

## 3. H2 and H3 · Position sizing in BOVA11

Signals are determined at the close. Positions enter at the next open and exit at that session's close. Regime sizing uses a weight of 1 minus 0.60 times the probability of high volatility.

In [ ]:
barras = bova.reindex(sessoes)
ret_oc = (barras.Close/barras.Open-1).shift(-1)
alto = probabilidades["HMM"].Alta
peso_regime = 1-MULTIPLICADOR_ALTO*alto
pesos = pd.DataFrame({"Fixo 100%":1.,"Peso por regime":peso_regime},index=sessoes)
for nome in ["Histórica 21","EWMA","HMM"]:
    pesos[f"Vol alvo · {nome}"] = (VOL_ALVO/np.sqrt(252*variancias[nome].clip(lower=1e-10))).clip(upper=1)
elegivel = data_alvo.ge(CORTE)&~data_alvo.isin(pd.to_datetime(list(QUARENTENA_BOVA11)))
dados = pesos.join(ret_oc.rename("ret")).loc[elegivel].dropna()
assert len(dados)>40

try:
    from backtest_intradiario import cvar_empirico, drawdown_capital, simular_intradiario
except ModuleNotFoundError as erro:
    if erro.name != "backtest_intradiario":
        raise
    # Fallback gerado a partir de backtest_intradiario.py.
    def drawdown_capital(capital):
        """Mede a queda desde o pico, incluindo o capital inicial unitário."""
        return capital / capital.cummax().clip(lower=1) - 1


    def cvar_empirico(retornos, cauda=0.05):
        """Média das piores perdas, com massa fracionária na fronteira."""
        valores = np.asarray(retornos, dtype=float)
        if valores.ndim != 1 or not valores.size or not np.isfinite(valores).all():
            raise ValueError("O CVaR exige um vetor de retornos finitos e não vazio.")
        if not 0 < cauda < 1:
            raise ValueError("A probabilidade de cauda deve ficar entre zero e um.")
        perdas = np.sort(-valores)[::-1]
        massa = len(perdas) * cauda
        inteiras = int(massa)
        return (perdas[:inteiras].sum() + (massa - inteiras) * perdas[inteiras]) / massa


    def simular_intradiario(pesos, retornos, custo_lado, *, detalhar=False):
        """Usa exposições e retornos do mesmo painel, inclusive dias em caixa."""
        if (pesos.empty or not pesos.index.equals(retornos.index)
            or not pesos.index.is_unique or not pesos.index.is_monotonic_increasing
            or not pesos.columns.is_unique):
            raise ValueError("Pesos e retornos devem compartilhar um painel cronológico sem duplicatas.")
        if not np.isfinite(custo_lado) or custo_lado < 0:
            raise ValueError("O custo por lado deve ser finito e não negativo.")
        if not np.isfinite(pesos.to_numpy(dtype=float)).all() or not np.isfinite(retornos).all():
            raise ValueError("Pesos e retornos devem ser finitos em todos os pregões.")

        bruto = pesos.mul(retornos, axis=0)
        custos = 2 * custo_lado * pesos.abs()
        liquido = bruto - custos
        capital = (1 + liquido).cumprod()
        desvio = liquido.std()
        exposicao = pesos.abs().mean()
        metricas = pd.DataFrame({
            "retorno_liquido": capital.iloc[-1] - 1,
            "vol_anual": desvio * np.sqrt(252),
            "sharpe": np.sqrt(252) * liquido.mean() / desvio.replace(0, np.nan),
            "drawdown": drawdown_capital(capital).min(),
            "VaR_95": -liquido.quantile(0.05),
            "CVaR_95": liquido.apply(cvar_empirico),
            "exposicao_media": exposicao,
            "turnover_medio": 2 * exposicao,
            "operacoes": pesos.ne(0).sum(),
        })
        if detalhar:
            metricas["retorno_bruto"] = (1 + bruto).prod() - 1
            metricas["soma_custos_diarios"] = custos.sum()
            metricas["n_pregoes"] = len(pesos)
        return metricas, capital, liquido

w = dados.drop(columns="ret").copy()
for nome in ["Peso por regime","Vol alvo · HMM"]:
    w[f"Fixo pareado · {nome}"] = w[nome].mean()
tabela_risco,curva_risco,_ = simular_intradiario(w,dados.ret,CUSTO_LADO)
display(tabela_risco)
tabela_risco.to_csv(PASTA/"trading_gestao_de_risco.csv")

registro = ler("sinal")
idx = registro.index.intersection(dados.index)
assert (pd.to_datetime(registro.loc[idx,"execucao"])==data_alvo.loc[idx]).all()
prev = registro.loc[idx,"previsao_bruta"]
lado = np.sign(prev).where(prev.abs()>.0035,0)
assert lado.ne(0).eq(registro.loc[idx,"sinal"]).all()
ponderado = lado*peso_regime.loc[idx]
constante = ponderado.abs().loc[lado.ne(0)].mean() if lado.ne(0).any() else 0.
sinais = pd.DataFrame({"Sinal congelado":lado,"Sinal + regime":ponderado,
                      "Sinal · escala fixa pareada":lado*constante},index=idx)
tabela_alpha,curva_alpha,_ = simular_intradiario(sinais,dados.ret.loc[idx],CUSTO_LADO)
display(tabela_alpha)
tabela_alpha.to_csv(PASTA/"trading_sinal_congelado.csv")
pd.concat({"pesos":sinais,"retorno":dados.ret.loc[idx].to_frame()},axis=1).to_csv(
    PASTA/"sinais_e_exposicao.csv",index_label="origem")
fig,ax = plt.subplots(2,1,figsize=(12,7))
curva_risco[["Fixo 100%","Peso por regime","Vol alvo · EWMA","Vol alvo · HMM"]].plot(
    ax=ax[0],title="BOVA11 · gestão de risco intradiária após custos")
curva_alpha.plot(ax=ax[1],title="XGBoost congelado · efeito de dimensionar a posição")
fig.tight_layout()
fig.savefig(PASTA/"trading_comparacao.png",dpi=140)
plt.show()

## 4. P1.2 · Regime stability before 2025

Compare two- and three-state HMM and Markov switching fits over two chronological windows of 63 targets each. Every fit uses only earlier returns.

In [ ]:
from itertools import combinations
import warnings

SEMENTES_ESTABILIDADE = [42,137,251]
TAMANHO_VALIDACAO = 63
alvos_validacao = treino.index[-2*TAMANHO_VALIDACAO:]
assert len(alvos_validacao)==2*TAMANHO_VALIDACAO and alvos_validacao.max()<CORTE
janelas_validacao = [alvos_validacao[:TAMANHO_VALIDACAO],alvos_validacao[TAMANHO_VALIDACAO:]]
suporta_rng = "rng" in inspect.signature(ms.fit).parameters

def ajustar_para_estabilidade(familia,k,semente,historico):
    valores = historico.to_numpy()
    if familia == "HMM":
        ajuste = GaussianHMM(n_components=k,covariance_type="diag",n_iter=400,
                             random_state=semente).fit(valores[:,None])
        param = (ajuste.means_.ravel(),ajuste.covars_.reshape(k,-1)[:,0],
                 ajuste.transmat_,ajuste.startprob_)
        convergiu = bool(ajuste.monitor_.converged and ajuste.monitor_.iter<ajuste.n_iter)
        return param,convergiu,ajuste.score(valores[:,None])
    modelo = MarkovRegression(valores,k_regimes=k,trend="c",
                             switching_trend=False,switching_variance=True)
    kwargs = {"rng":np.random.default_rng(semente),"search_reps":8} if suporta_rng else {}
    ajuste = modelo.fit(em_iter=8,maxiter=400,disp=False,**kwargs)
    coef = dict(zip(modelo.param_names,ajuste.params))
    param = (np.repeat(coef["const"],k),np.array([coef[f"sigma2[{i}]"] for i in range(k)]),
             modelo.regime_transition_matrix(ajuste.params)[:,:,0].T,
             modelo.initial_probabilities(ajuste.params))
    return param,bool(ajuste.mle_retvals["converged"]),ajuste.llf

ajustes,estados_estabilidade,comparacoes = [],[],[]
prob_por_semente = {}
for janela,alvos in enumerate(janelas_validacao,start=1):
    origens = sessoes[sessoes.get_indexer(alvos)-1]
    assert data_alvo.loc[origens].tolist()==alvos.tolist()
    hist = treino.loc[treino.index<alvos.min()]
    assert hist.index.max()<=origens.min()
    serie = (r.loc[hist.index.min():alvos.max()]*100)
    pos = serie.index.get_indexer(origens)
    pos_alvo = serie.index.get_indexer(alvos)
    assert (pos>=0).all() and (pos_alvo>=0).all() and (pos_alvo==pos+1).all()
    observado_pct = serie.loc[alvos].to_numpy()
    assert np.isfinite(observado_pct).all()
    for familia in ["HMM","Markov Switching"]:
        sementes = SEMENTES_ESTABILIDADE if familia=="HMM" or suporta_rng else [SEED]
        for k in [2,3]:
            grupo = (janela,familia,k)
            prob_por_semente[grupo] = {}
            for semente in sementes:
                linha = {"janela":janela,"modelo":familia,"estados":k,"semente":semente,
                         "n_ajuste":len(hist),"fim_ajuste":hist.index.max().date(),
                         "inicio_alvo":alvos.min().date(),"fim_alvo":alvos.max().date(),
                         "n_validacao":len(alvos)}
                try:
                    with warnings.catch_warnings(record=True) as avisos:
                        warnings.simplefilter("always")
                        param,convergiu,llf = ajustar_para_estabilidade(familia,k,semente,hist)
                        atual,futura,medias,var,A = filtrar(serie,*param)
                    p = futura[pos]
                    q = atual[pos_alvo]
                    h = np.maximum(p@(var+medias**2)/1e4,1e-10)
                    perdas = np.log(h)+(observado_pct/100)**2/h
                    logp = logsumexp(np.log(np.maximum(p,1e-300))+
                                      norm.logpdf(observado_pct[:,None],medias,np.sqrt(var)),axis=1)
                    if not np.isfinite(np.r_[perdas,logp,llf]).all():
                        raise ValueError("Métricas não finitas.")
                    linha.update({"convergiu":convergiu,"loglik_ajuste":llf,
                                  "QLIKE":perdas.mean(),"neg_log_score_pct":-logp.mean(),
                                  "alertas_ajuste":"; ".join(sorted({str(a.message) for a in avisos}))})
                    if convergiu:
                        prob_por_semente[grupo][semente] = p[:,-1]
                    for i in range(k):
                        diagonal = float(A[i,i])
                        ocupacao = float(q[:,i].mean())
                        dominancia = float((q.argmax(axis=1)==i).mean())
                        razao = float(var[i]/var[i-1]) if i else np.nan
                        estados_estabilidade.append({**{chave:linha[chave] for chave in
                            ["janela","modelo","estados","semente"]},"ordem_variancia":i+1,
                            "convergiu":convergiu,"vol_anual":np.sqrt(var[i]*252)/100,
                            "ocupacao_filtrada":ocupacao,"frequencia_dominante":dominancia,
                            "duracao_teorica":1/(1-diagonal) if diagonal<1 else np.inf,
                            "razao_variancia_anterior":razao,
                            "alerta_ocupacao":ocupacao<.05 or dominancia==0,
                            "alerta_separacao":bool(i and razao<1.20)})
                except (ValueError,RuntimeError,AssertionError,np.linalg.LinAlgError) as erro:
                    linha.update({"convergiu":False,"alertas_ajuste":str(erro)})
                ajustes.append(linha)
for (janela,familia,k),por_semente in prob_por_semente.items():
    for (sa,pa),(sb,pb) in combinations(por_semente.items(),2):
        comparacoes.append({"janela":janela,"modelo":familia,"estados":k,
                            "semente_a":sa,"semente_b":sb,
                            "dif_abs_media_p_alta":np.abs(pa-pb).mean()})

ajustes = pd.DataFrame(ajustes)
estados_estabilidade = pd.DataFrame(estados_estabilidade)
divergencias = pd.DataFrame(comparacoes,columns=[
    "janela","modelo","estados","semente_a","semente_b","dif_abs_media_p_alta"])
ajustes.to_csv(PASTA/"estabilidade_ajustes.csv",index=False)
estados_estabilidade.to_csv(PASTA/"estabilidade_estados.csv",index=False)
divergencias.to_csv(PASTA/"estabilidade_entre_sementes.csv",index=False)
chaves = ["janela","modelo","estados"]
resumo_estabilidade = ajustes.groupby(chaves).convergiu.agg(
    tentativas="size",convergencias="sum").reset_index()
valido = ajustes.loc[ajustes.convergiu]
if not valido.empty:
    desempenho = valido.groupby(chaves).agg(
        QLIKE_mediano=("QLIKE","median"),QLIKE_min=("QLIKE","min"),QLIKE_max=("QLIKE","max"),
        neg_log_score_mediano=("neg_log_score_pct","median")).reset_index()
    resumo_estabilidade = resumo_estabilidade.merge(desempenho,on=chaves,how="left")
if not divergencias.empty:
    divergencia = divergencias.groupby(chaves).dif_abs_media_p_alta.max().rename(
        "maior_dif_media_p_alta").reset_index()
    resumo_estabilidade = resumo_estabilidade.merge(divergencia,on=chaves,how="left")
display(resumo_estabilidade)
resumo_estabilidade.to_csv(PASTA/"estabilidade_resumo.csv",index=False)

## 5. P1.3 · Walk-forward refits and observable risk

Compare a two-state Markov switching model fitted before 2025 with a version refitted every 63 origins when enough contiguous observations are available.

In [ ]:
PASSO_REAJUSTE = 63
MIN_RETORNOS_REAJUSTE = 126
LIMIAR_QUANTIL = .90

origens_avaliar = sessoes[data_alvo.ge(CORTE).to_numpy()]
assert len(origens_avaliar)>=PASSO_REAJUSTE
corte_extremo = float(treino.abs().quantile(LIMIAR_QUANTIL))  # percentual
frequencia_ajuste = float(treino.abs().gt(corte_extremo).mean())
param_fixo,convergiu,llf_fixo = ajustar_para_estabilidade("Markov Switching",2,SEED,treino)
if not convergiu:
    raise RuntimeError("O Markov Switching fixo de dois estados não convergiu.")

def prever_risco_em_extremo(param,origens):
    # A trajetória até a última origem contém somente retornos já disponíveis.
    serie = r.loc[:origens[-1]]*100
    _,futura,medias,vars_,_ = filtrar(serie,*param)
    pos = serie.index.get_indexer(origens)
    assert (pos>=0).all()
    p = futura[pos]
    dispersao = np.sqrt(vars_)
    evento_por_estado = norm.cdf(-corte_extremo,medias,dispersao)+norm.sf(corte_extremo,medias,dispersao)
    return pd.DataFrame({"variancia":p@(vars_+medias**2)/1e4,
                         "prob_extremo":p@evento_por_estado},index=origens)

fixo = prever_risco_em_extremo(param_fixo,origens_avaliar)
param_corrente = param_fixo
fim_corrente = treino.index.max()
partes_reajuste,log_reajuste = [],[]
for i in range(0,len(origens_avaliar),PASSO_REAJUSTE):
    bloco = origens_avaliar[i:i+PASSO_REAJUSTE]
    disponivel = r.loc[:bloco[0]]
    ultimo_bloco = disponivel.groupby(disponivel.isna().cumsum()).get_group(
        disponivel.isna().sum()).dropna()*100
    status = "histórico_contínuo_insuficiente"
    detalhe = ""
    llf = np.nan
    if len(ultimo_bloco)>=MIN_RETORNOS_REAJUSTE:
        if ultimo_bloco.index.max()<=fim_corrente:
            status = "ajuste_atual_já_usa_o_histórico"
        else:
            try:
                novo,ok,llf = ajustar_para_estabilidade("Markov Switching",2,SEED,ultimo_bloco)
                # Valida parâmetros e filtro antes de promover o ajuste.
                if ok:
                    prever_risco_em_extremo(novo,bloco[:1])
                    param_corrente,fim_corrente = novo,ultimo_bloco.index.max()
                    status = "reajustado"
                else:
                    status = "sem_convergência"
            except (ValueError,RuntimeError,AssertionError,np.linalg.LinAlgError) as erro:
                status,detalhe = "falha_no_ajuste",str(erro)
    assert fim_corrente<=bloco[0]
    p = prever_risco_em_extremo(param_corrente,bloco)
    p["fim_ajuste"] = fim_corrente
    partes_reajuste.append(p)
    log_reajuste.append({"primeira_origem":bloco[0],"ultima_origem":bloco[-1],
        "n_contiguo_disponivel":len(ultimo_bloco),"fim_ajuste_usado":fim_corrente,
        "status":status,"loglik_tentativa":llf,"detalhe":detalhe})
reajuste = pd.concat(partes_reajuste)
pd.DataFrame(log_reajuste).to_csv(PASTA/"walk_forward_reajustes.csv",index=False)

ewma = variancias.EWMA.reindex(origens_avaliar)
prob_ewma = 2*norm.sf(corte_extremo/100/np.sqrt(ewma.clip(lower=1e-10)))
retorno_alvo = r.shift(-1).reindex(origens_avaliar)
quadro = pd.DataFrame({"data_alvo":data_alvo.loc[origens_avaliar],
    "retorno_alvo":retorno_alvo,"variancia_observada":retorno_alvo.pow(2),
    "variancia_fixo":fixo.variancia,"variancia_reajuste":reajuste.variancia,
    "variancia_ewma":ewma,"prob_fixo":fixo.prob_extremo,"prob_reajuste":reajuste.prob_extremo,
    "prob_ewma":prob_ewma,"prob_frequencia":frequencia_ajuste,
    "fim_ajuste_fixo":treino.index.max(),"fim_ajuste_reajuste":reajuste.fim_ajuste})
quadro["evento"] = retorno_alvo.abs().gt(corte_extremo/100).where(retorno_alvo.notna())
quadro.to_csv(PASTA/"walk_forward_previsoes.csv",index_label="origem")
amostra = quadro.dropna()
assert len(amostra)>100
assert (amostra.fim_ajuste_reajuste<=amostra.index).all()

# Fallback gerado a partir de risco_estatistico.py para uso independente no Colab.
try:
    from risco_estatistico import adjacencia_sessoes, intervalo_media_pareada
except ModuleNotFoundError as erro:
    if erro.name!="risco_estatistico":
        raise
    def adjacencia_sessoes(sessoes, datas):
        """Marca vizinhos de calendário sem unir os dois lados de uma lacuna."""
        posicoes = sessoes.get_indexer(datas)
        passos = np.diff(posicoes)
        if len(posicoes)==0 or (posicoes<0).any() or (passos<=0).any():
            raise ValueError("A amostra precisa de sessões conhecidas, únicas e em ordem.")
        return passos==1

    def intervalo_media_pareada(diferenca,adjacentes,bloco=10,rep=3000,semente=42):
        """Bootstrap circular dentro de cada trecho contínuo, com tamanho preservado."""
        diferenca = np.asarray(diferenca,dtype=float)
        adjacentes = np.asarray(adjacentes,dtype=bool)
        if (diferenca.ndim!=1 or not len(diferenca) or not np.isfinite(diferenca).all()
            or adjacentes.shape!=(len(diferenca)-1,) or bloco<1 or rep<1):
            raise ValueError("Amostra pareada, adjacência ou configuração inválida.")
        rng = np.random.default_rng(semente)
        cortes = np.flatnonzero(~adjacentes)+1
        somas = np.zeros(rep)
        for seg in np.split(diferenca,cortes):
            k = min(bloco,len(seg))
            inicios = rng.integers(len(seg),size=(rep,int(np.ceil(len(seg)/k))))
            indices = ((inicios[:,:,None]+np.arange(k))%len(seg)).reshape(rep,-1)
            somas += seg[indices[:,:len(seg)]].sum(axis=1)
        return np.quantile(somas/len(diferenca),[.025,.975])

def intervalo_pareado(diferenca,bloco=10,n_boot=3000):
    adjacentes = adjacencia_sessoes(sessoes,diferenca.index)
    return intervalo_media_pareada(diferenca,adjacentes,bloco,n_boot,SEED)

def qlike(h):
    h = h.clip(lower=1e-10)
    return np.log(h)+amostra.variancia_observada/h

base_risco = qlike(amostra.variancia_ewma)
linhas_risco = []
for nome in ["fixo","reajuste","ewma"]:
    perda = qlike(amostra[f"variancia_{nome}"])
    dif = perda-base_risco
    lo,hi = intervalo_pareado(dif)
    linhas_risco.append({"modelo":nome,"n":len(perda),"QLIKE":perda.mean(),
        "dif_vs_ewma":dif.mean(),"IC_inf":lo,"IC_sup":hi})
tabela_walk = pd.DataFrame(linhas_risco).sort_values("QLIKE")
display(tabela_walk)
tabela_walk.to_csv(PASTA/"walk_forward_qlike.csv",index=False)

y_evento = amostra.evento.astype(int)
linhas_evento,faixas = [],[]
for nome in ["fixo","reajuste","ewma","frequencia"]:
    p = amostra[f"prob_{nome}"].clip(1e-10,1-1e-10)
    brier = (p-y_evento)**2
    perda_log = -(y_evento*np.log(p)+(1-y_evento)*np.log1p(-p))
    dif_brier = brier-(amostra.prob_frequencia-y_evento)**2
    lo,hi = intervalo_pareado(dif_brier)
    linhas_evento.append({"modelo":nome,"n":len(y_evento),"eventos":int(y_evento.sum()),
        "Brier":brier.mean(),"log_loss":perda_log.mean(),"prob_media":p.mean(),
        "frequencia_observada":y_evento.mean(),"dif_brier_vs_frequencia":dif_brier.mean(),
        "IC_inf":lo,"IC_sup":hi})
    grupos = pd.DataFrame({"p":p,"evento":y_evento}).groupby(
        pd.cut(p,[0,.05,.10,.20,.40,1.],include_lowest=True),observed=True)
    faixa = grupos.agg(n=("evento","size"),prob_media=("p","mean"),
                      frequencia_observada=("evento","mean")).reset_index(names="faixa")
    faixas.append(faixa.assign(modelo=nome))
calibracao = pd.DataFrame(linhas_evento).sort_values("Brier")
display(calibracao)
calibracao.to_csv(PASTA/"calibracao_evento_extremo.csv",index=False)
pd.concat(faixas,ignore_index=True).to_csv(PASTA/"calibracao_faixas_evento_extremo.csv",index=False)
(PASTA/"config_walk_forward.json").write_text(json.dumps({
    "revisao":"P1.3","passo_reajuste":PASSO_REAJUSTE,"min_retornos_reajuste":MIN_RETORNOS_REAJUSTE,
    "evento_quantile_treino":LIMIAR_QUANTIL,"limiar_absoluto_logret":corte_extremo/100,
    "frequencia_treino":frequencia_ajuste,"fim_treino_fixo":str(treino.index.max().date()),
    "semente":SEED,"bootstrap_bloco":10},indent=2),encoding="utf-8")
print(f"Evento extremo: |retorno log| > {corte_extremo/100:.2%} | {int(y_evento.sum())}/{len(y_evento)} na amostra comum")

## 6. P1.7 · Execution and risk benchmarks

All intraday policies receive the same BOVA11 open-to-close returns, with exposure capped at 100% and a 15-basis-point cost per side. The passive benchmark uses its own holding period and cost convention.

In [ ]:
JANELA_PESO_FIXO = 126
# O mesmo dia problemático do contrato fica indisponível para OHLC.
oc = (barras.Close/barras.Open-1).where(~barras.index.isin(pd.to_datetime(list(QUARENTENA_BOVA11))))
var_oc = oc.pow(2).groupby(oc.isna().cumsum()).transform(
    lambda s:s.ewm(alpha=.06,adjust=False,min_periods=21).mean()).where(oc.notna())
variancias_h2 = pd.DataFrame({"EWMA fechamento":variancias.EWMA,"EWMA intradiário":var_oc})
pesos_h2 = (VOL_ALVO/np.sqrt(252*variancias_h2.clip(lower=1e-10))).clip(upper=1)
alvo_oc = oc.shift(-1)
comum_h2 = pesos_h2.join(alvo_oc.rename("ret")).dropna()
pre_h2 = comum_h2.loc[data_alvo.loc[comum_h2.index].lt(CORTE)].tail(JANELA_PESO_FIXO)
assert len(pre_h2)==JANELA_PESO_FIXO
amostra_h2 = comum_h2.loc[data_alvo.loc[comum_h2.index].ge(CORTE)]
assert len(amostra_h2)>100
assert data_alvo.loc[pre_h2.index].max()<data_alvo.loc[amostra_h2.index].min()
w_h2 = amostra_h2.drop(columns="ret").copy()
w_h2["Fixo 100%"] = 1.
for nome in variancias_h2:
    w_h2[f"Fixo pré-teste · {nome}"] = pre_h2[nome].mean()

h2,curvas_h2,liquidos_h2 = simular_intradiario(w_h2,amostra_h2.ret,CUSTO_LADO,detalhar=True)
h2["peso_calibrado_pre_teste"] = [pre_h2[c.split(" · ")[1]].mean() if c.startswith("Fixo pré-teste") else np.nan
                                  for c in h2.index]
display(h2)
h2.to_csv(PASTA/"H2_intradiario_benchmarks.csv",index_label="politica")
curvas_h2.assign(data_alvo=data_alvo.loc[curvas_h2.index]).to_csv(PASTA/"H2_curvas_intradiario.csv",index_label="origem")
comparacao_h2 = []
for nome,estimativa in variancias_h2.loc[amostra_h2.index].items():
    h = estimativa.clip(lower=1e-10)
    q = (np.log(h)+amostra_h2.ret.pow(2)/h).mean()
    diferenca = liquidos_h2[nome]-liquidos_h2[f"Fixo pré-teste · {nome}"]
    lo,hi = intervalo_pareado(diferenca)
    comparacao_h2.append({"modelo":nome,"QLIKE_oc":q,"n":len(amostra_h2),
        "ganho_liquido_diario_vs_fixo_pre_teste":diferenca.mean(),"IC_inf":lo,"IC_sup":hi})
comparacao_h2 = pd.DataFrame(comparacao_h2)
display(comparacao_h2)
comparacao_h2.to_csv(PASTA/"H2_previsao_e_controles.csv",index=False)

passivo_preco = bova["Adj Close"].reindex(sessoes[sessoes>=CORTE])
assert len(passivo_preco.dropna())>=2
entrada,saida = passivo_preco.first_valid_index(),passivo_preco.last_valid_index()
passivo_preco = passivo_preco.loc[entrada:saida]
passivo_bruto = passivo_preco/passivo_preco.loc[entrada]
passivo_liquido = passivo_bruto*(1-CUSTO_LADO)
passivo_liquido.loc[saida] *= 1-CUSTO_LADO
ret_dia = passivo_liquido.pct_change(fill_method=None)
ret_dia.loc[entrada] = -CUSTO_LADO
observados_passivo = ret_dia.dropna().to_numpy()
cvar = cvar_empirico(observados_passivo)
passivo = pd.DataFrame([{"entrada_fechamento":entrada,"saida_fechamento":saida,
    "n_sessoes":len(passivo_preco),"retornos_diarios_observados":len(observados_passivo),
    "retorno_bruto":passivo_bruto.loc[saida]-1,"retorno_liquido":passivo_liquido.loc[saida]-1,
    "impacto_custos_no_retorno":passivo_bruto.loc[saida]-passivo_liquido.loc[saida],
    "drawdown":drawdown_capital(passivo_liquido).min(),
    "CVaR_95":cvar,"vol_anual":ret_dia.std()*np.sqrt(252),
    "sharpe":np.sqrt(252)*ret_dia.mean()/ret_dia.std() if ret_dia.std()>0 else np.nan,
    "exposicao_media":1.,"turnover_medio":2/len(passivo_preco),"operacoes":1}])
display(passivo)
passivo.to_csv(PASTA/"H2_benchmark_passivo.csv",index=False)
pd.DataFrame({"capital_bruto":passivo_bruto,"capital_liquido":passivo_liquido}).to_csv(
    PASTA/"H2_curva_passiva.csv",index_label="data")

# Preserva todas as oportunidades e o lado do sinal congelado.
ret_h3 = dados.ret.loc[sinais.index]
h3,curvas_h3,liquidos_h3 = simular_intradiario(sinais,ret_h3,CUSTO_LADO,detalhar=True)
h3.to_csv(PASTA/"H3_execucao_congelada.csv",index_label="politica")
ativos_h3 = sinais["Sinal congelado"].ne(0)
if ativos_h3.any():
    operados = sinais.loc[ativos_h3]
    x = operados["Sinal + regime"].abs().to_numpy()
    lados = operados["Sinal congelado"].to_numpy()
    r_h3 = ret_h3.loc[ativos_h3].to_numpy()
    rng = np.random.default_rng(SEED)
    permutados = np.tile(x,(5000,1))
    for lado_controle in [-1,1]:
        mascara = lados==lado_controle
        permutados[:,mascara] = [rng.permutation(x[mascara]) for _ in range(len(permutados))]
    controle = np.prod(1+permutados*(lados*r_h3-2*CUSTO_LADO),axis=1)-1
    q = np.quantile(controle,[.025,.5,.975])
    diferenca = liquidos_h3["Sinal + regime"]-liquidos_h3["Sinal · escala fixa pareada"]
    lo,hi = intervalo_pareado(diferenca)
    observado_h3 = h3.loc["Sinal + regime","retorno_liquido"]
    tabela_h3 = pd.DataFrame([{"operacoes":int(ativos_h3.sum()),"retorno_regime":observado_h3,
        "controle_q025":q[0],"controle_mediana":q[1],"controle_q975":q[2],
        "percentil_descritivo":(controle<=observado_h3).mean(),
        "diferenca_diaria_vs_fixo_pareado":diferenca.mean(),"IC_inf":lo,"IC_sup":hi}])
    display(tabela_h3)
    tabela_h3.to_csv(PASTA/"H3_controle_por_lado.csv",index=False)
fig,axs = plt.subplots(2,1,figsize=(12,7))
curvas_h2[["EWMA fechamento","EWMA intradiário","Fixo 100%"]].set_axis(
    pd.DatetimeIndex(data_alvo.loc[curvas_h2.index])).plot(ax=axs[0],title="Operações diárias · após custos")
passivo_liquido.plot(ax=axs[1],title="BOVA11 passivo · capital observado após custos")
fig.tight_layout()
fig.savefig(PASTA/"H2_benchmarks.png",dpi=140)
plt.show()
(PASTA/"config_H2_H3.json").write_text(json.dumps({
    "revisao":"P1.7","janela_peso_fixo":JANELA_PESO_FIXO,"fim_calibracao_peso":str(pre_h2.index.max().date()),
    "vol_alvo":VOL_ALVO,"custo_lado":CUSTO_LADO,"lambda_ewma":.94,"semente":SEED,
    "permutacoes_por_lado":5000},indent=2),encoding="utf-8")
print("Resultados para download:",shutil.make_archive(str(PASTA),"zip",PASTA))
